# carGO PH — Blockchain Integration
**Course:** MO-IT148 — Application Development and Emerging Technologies <br>
**Group:** NodeBlk <br><br><br>
**Week:** 4-5 — Blockchain Ledger Submission <br>
**Description:** End-to-end Web3.py pipeline that connects to a local Ganache instance, loads a compiled Solidity contract, and bulk-writes IoT sensor records (GPS, temperature, RFID) from CSV files onto the blockchain via type-specific helper functions. Includes on-chain verification by reading back live contract counters and retrieving stored records across separate data-type arrays.<br>

## Section 1 — Ganache Connection

In [2]:
from web3 import Web3
import pandas as pd
import time

ganache_url = "http://127.0.0.1:7545"  # Ganache RPC port
web3 = Web3(Web3.HTTPProvider(ganache_url))  # HTTPProvider connects over HTTP to local Ganache

if not web3.is_connected():  # is_connected() returns False if Ganache is unreachable
    raise ConnectionError("Cannot connect to Ganache. Ensure Ganache is running on port 7545.")

print("Connected to Ganache:", web3.is_connected())
print("Latest block number:", web3.eth.block_number)  # confirms live blockchain data is reachable

Connected to Ganache: True
Latest block number: 1


## Section 2 — Contract Load

In [3]:
import json

with open("../contracts/IoTDataStorage_compData.json") as f:  # load latest Remix compilation artifact
    artifact = json.load(f)

contract_config = {
    "address": "0x2728C1070A6a3E4c42A82537Db99AA828F3E01aA",  # update after each Remix redeployment
    "abi": artifact["abi"]  # ABI extracted from compiled artifact
}

iot_contract = web3.eth.contract(
    address=web3.to_checksum_address(contract_config["address"]),  # checksummed format required by Web3.py
    abi=contract_config["abi"]
)

web3.eth.default_account = web3.eth.accounts[0]  # accounts[0] is the deployer — matches the contract owner

print("Contract loaded at:", contract_config["address"])
print("Default account:", web3.eth.default_account)

pre_write_count = iot_contract.functions.iotRecordCount().call()  # .call() reads without a transaction — iotRecordCount() is the actual contract function
print("Records before write:", pre_write_count)

Contract loaded at: 0x2728C1070A6a3E4c42A82537Db99AA828F3E01aA
Default account: 0x2cF359c0E136B5F9f55b3748D40AC2C90b43249E
Records before write: 0


## Section 3 — Dummy Test Transaction

In [4]:
# Step 1 — register a test shipment (required before any storeData() call)
reg_tx_hash = iot_contract.functions.registerShipment(
    "TEST-001",          # rfidTag — unique identifier for this test shipment
    0,                   # GoodsCategory enum index: 0 = DeepFreeze
    "Manila",            # origin
    "Cebu",              # destination
    1,                   # packageCount
    "VH-TEST",           # vehicleId
    "DR-TEST"            # driverId
).transact()  # .transact() writes to blockchain — returns tx_hash as HexBytes

web3.eth.wait_for_transaction_receipt(reg_tx_hash)  # block until shipment registration is mined
print("Test shipment registered. TX:", reg_tx_hash.hex())  # .hex() converts HexBytes to readable string

# Step 2 — store a dummy IoT record against the registered shipment
tx_hash = iot_contract.functions.storeData(
    "READ-TEST-001",     # readingId
    "TEST-001",          # rfidTag — must match the registered shipment above
    "DEV-TEST",          # deviceId
    "GPS",               # deviceType
    "latitude",          # dataType
    "14.5995"            # dataValue
).transact()  # .transact() writes to blockchain — returns tx_hash as HexBytes

receipt = web3.eth.wait_for_transaction_receipt(tx_hash)  # block until transaction is mined
print("Dummy record stored. TX:", tx_hash.hex())
print("Transaction status:", receipt.status)  # 1 = success, 0 = failed

# Step 3 — read back the dummy record to confirm it landed on-chain
dummy_record = iot_contract.functions.iotRecords(0).call()  # public array auto-getter — index 0 = first record
print("Record at index 0:", dummy_record)

Test shipment registered. TX: 40b52da68cd5fc63f81d1111ee06db87d865de30b18d1f228b862497b5bd93cd
Dummy record stored. TX: 0f49a052b9866be4dc23368f5229254e1b220d6587d1524e087c3805bc3483c3
Transaction status: 1
Record at index 0: [1780590050, 'READ-TEST-001', 'TEST-001', 'DEV-TEST', 'GPS', 'latitude', '14.5995', '0x2cF359c0E136B5F9f55b3748D40AC2C90b43249E']


## Section 3b — Type-Specific Record Helpers

In [5]:
def store_gps_record(rfid_tag, device_id, data_value):
    """Store a GPS location record on-chain via storeGPS()."""
    lat, lng = data_value.split(",")  # splits "14.60,120.98" into two variables in one line
    lat = lat.strip()                  # removes any accidental whitespace around the value
    lng = lng.strip()
    tx = iot_contract.functions.storeGPS(
        str(rfid_tag), str(device_id), lat, lng
    ).transact()
    return web3.eth.wait_for_transaction_receipt(tx)

def store_temperature_record(rfid_tag, device_id, data_value):
    """Store a temperature reading on-chain via storeTemperature()."""
    temp_int = int(float(data_value) * 10)  # "12.2" → 12.2 → 122.0 → 122 — int16-safe encoding
    tx = iot_contract.functions.storeTemperature(
        str(rfid_tag), str(device_id), temp_int
    ).transact()
    return web3.eth.wait_for_transaction_receipt(tx)

def store_rfid_record(rfid_tag, device_id, data_value):
    """Store an RFID scan status on-chain via storeRFIDScan()."""
    tx = iot_contract.functions.storeRFIDScan(
        str(rfid_tag), str(device_id), str(data_value)
    ).transact()
    return web3.eth.wait_for_transaction_receipt(tx)

def store_generic_record(reading_id, rfid_tag, device_id, data_type, data_value):
    """Fallback: store any unrecognized data type via storeData()."""
    tx = iot_contract.functions.storeData(
        str(reading_id), str(rfid_tag), str(device_id),
        str(data_type), str(data_type), str(data_value)
    ).transact()
    return web3.eth.wait_for_transaction_receipt(tx)

## Section 3c — CSV Data Preview

In [6]:
import pandas as pd

preview_df = pd.read_csv("../data/iot_data.csv")
print(f"Total records: {len(preview_df)}")
print("\nFirst 3 records:")
print(preview_df.head(3).to_string())

Total records: 329

First 3 records:
     reading_id  rfid_tag device_id    data_type            data_value            timestamp    gps_lat     gps_lng  temperature_c
0  RDG-TMP-0031  RFID-011    TMP104  Temperature                  12.2  2026-05-03 07:00:00        NaN         NaN           12.2
1  RDG-GPS-0096  RFID-020    GPS291          GPS  14.601956,120.989036  2026-05-03 07:00:00  14.601956  120.989036            NaN
2  RDG-GPS-0051  RFID-011    GPS728          GPS  14.620032,120.962165  2026-05-03 07:00:00  14.620032  120.962165            NaN


## Section 4 — CSV Load + Bulk Write

In [7]:
CATEGORY_MAP = {
    "Deep Freeze": 0, "Frozen": 1, "Chill/Refrigerated": 2,
    "Pharma": 3, "Cool-Chain": 4, "Dry Goods": 5,
    "Electronics": 6, "Clothing": 7, "Industrial": 8
}

def run_bulk_write():
    """Register all shipments from CSV, then write all IoT records with type routing."""
    # --- Shipment Registration ---
    ship_df = pd.read_csv("../data/shipment_registry.csv")
    print(f"Registering {len(ship_df)} shipments...")
    for _, ship in ship_df.iterrows():
        reg_hash = iot_contract.functions.registerShipment(
            ship["rfid_tag"],
            CATEGORY_MAP[ship["goods_category"]],
            ship["origin"], ship["destination"],
            int(ship["package_count"]),
            ship["vehicle_id"], ship["driver_id"]
        ).transact()
        web3.eth.wait_for_transaction_receipt(reg_hash)
        time.sleep(0.5)  # 0.5s — down from 1.0s, still prevents nonce conflicts
    print("All shipments registered.\n")

    # --- IoT Bulk Write ---
    iot_df = pd.read_csv("../data/iot_data.csv")
    print(f"Loaded {len(iot_df)} IoT records. Starting bulk write...")

    for index, row in iot_df.iterrows():
        try:
            data_type  = row["data_type"]
            rfid_tag   = row["rfid_tag"]
            device_id  = row["device_id"]
            data_value = str(row["data_value"])

            if data_type == "GPS":
                receipt = store_gps_record(rfid_tag, device_id, data_value)
            elif data_type == "Temperature":
                receipt = store_temperature_record(rfid_tag, device_id, data_value)
            elif data_type == "RFID":
                receipt = store_rfid_record(rfid_tag, device_id, data_value)
            else:
                receipt = store_generic_record(
                    row["reading_id"], rfid_tag, device_id, data_type, data_value
                )

            print(f"{data_type} | {rfid_tag} | {data_value} | Txn: {receipt.transactionHash.hex()}")
            # receipt.transactionHash is HexBytes — .hex() converts it to a readable string
            time.sleep(0.5)

        except Exception as e:  # catches any error for this row — loop continues instead of stopping
            print(f"Row {index} failed: {e}")

    print("\nBulk write complete.")

run_bulk_write()

Registering 30 shipments...
All shipments registered.

Loaded 329 IoT records. Starting bulk write...
Temperature | RFID-011 | 12.2 | Txn: 25bc4e308f9c8a353dd296f6e2341694db140cda9122e64fe4c65cb80ce8d058
GPS | RFID-020 | 14.601956,120.989036 | Txn: e73c13cd7e05e80fec580bad90421086bf4acd183c88e97668d4bd54de408a5d
GPS | RFID-011 | 14.620032,120.962165 | Txn: 6046021dffc814426cf930e980e325b831e1d1b21b4549e5871d4a01c3480356
RFID | RFID-011 | VERIFIED | Txn: bc98aee705ea2cf7ff025d569cf4351d0b4194d9e7c7a95b00dae2872396fc73
GPS | RFID-020 | 14.595757,120.981906 | Txn: 52dd4c2233c27da1094a65092fa7b0f494b3f58690386289a9a1920883f7261e
GPS | RFID-028 | 14.374937,121.038127 | Txn: 7fc3708409bf4433a2f67abdfaa2fb5500bd5a18f2c0de3a95466ecaf4a54268
RFID | RFID-020 | VERIFIED | Txn: 2a81e10fc06760449e22ace73ddede50aa5b625475dfa406b1b76e210d77a744
Temperature | RFID-011 | 13.6 | Txn: 063248cbae3e01d95b4b84d42253db8a3b8a7f6f0ef55248981af479d3eb894e
GPS | RFID-011 | 14.622406,120.970944 | Txn: a0e43bec0d7

## Section 5 — Verification

In [8]:
def run_verification():
    """Read all 5 on-chain counters and verify the first real shipment and IoT record."""
    print("=== On-Chain Record Counts ===")
    print(f"  Shipments registered : {iot_contract.functions.shipmentCount().call()}")
    print(f"  Generic IoT records  : {iot_contract.functions.iotRecordCount().call()}")
    print(f"  GPS records          : {iot_contract.functions.gpsRecordCount().call()}")
    print(f"  Temperature records  : {iot_contract.functions.tempRecordCount().call()}")
    print(f"  RFID scan records    : {iot_contract.functions.rfidRecordCount().call()}")

    # First REAL shipment — index [1] skips TEST-001 dummy
    first_tag = iot_contract.functions.getAllRFIDTags().call()[1]
    shipment = iot_contract.functions.getShipment(first_tag).call()
    print(f"\n=== First Real Shipment on Chain ({first_tag}) ===")
    print(f"  Origin      : {shipment[2]}")
    print(f"  Destination : {shipment[3]}")
    print(f"  Category    : {shipment[1]}")
    print(f"  Vehicle ID  : {shipment[5]}")
    print(f"  Registered  : {shipment[7]}")

    # First REAL IoT record from iot_data.csv
    # Read first row of CSV to know which type it is
    iot_df = pd.read_csv("../data/iot_data.csv")
    first_type = iot_df.iloc[0]["data_type"]  # check what type first record is

    # Then call the correct blockchain array based on that type
    if first_type == "Temperature":
        record = iot_contract.functions.tempRecords(0).call() # blockchain call
        print(f"\n=== First Real IoT Record on Chain ===")
        print(f"  Timestamp  : {record[0]}")
        print(f"  RFID Tag   : {record[1]}")
        print(f"  Device ID  : {record[2]}")
        print(f"  Temperature: {record[3] / 10}°C")
    elif first_type == "GPS":
        record = iot_contract.functions.gpsRecords(0).call()
        print(f"\n=== First Real IoT Record on Chain ===")
        print(f"  Timestamp  : {record[0]}")
        print(f"  RFID Tag   : {record[1]}")
        print(f"  Device ID  : {record[2]}")
        print(f"  Latitude   : {record[3]}")
        print(f"  Longitude  : {record[4]}")
    elif first_type == "RFID":
        record = iot_contract.functions.rfidRecords(0).call()
        print(f"\n=== First Real IoT Record on Chain ===")
        print(f"  Timestamp  : {record[0]}")
        print(f"  RFID Tag   : {record[1]}")
        print(f"  Device ID  : {record[2]}")
        print(f"  Status     : {record[3]}")

run_verification()

=== On-Chain Record Counts ===
  Shipments registered : 31
  Generic IoT records  : 1
  GPS records          : 150
  Temperature records  : 95
  RFID scan records    : 84

=== First Real Shipment on Chain (RFID-001) ===
  Origin      : General Mariano Alvarez
  Destination : San Andres
  Category    : 3
  Vehicle ID  : VH-001
  Registered  : 1780590080

=== First Real IoT Record on Chain ===
  Timestamp  : 1780590105
  RFID Tag   : RFID-011
  Device ID  : TMP104
  Temperature: 12.2°C
